# Notebook 12: Transformers
**Estimated time:** 50-60 min
**Prerequisites:** Notebooks 1-11

---
## What you will learn
1. The complete Transformer architecture (Attention Is All You Need, 2017)
2. Multi-Head Attention — why multiple heads?
3. Positional Encoding — giving transformers a sense of position
4. The Encoder block: attention + feed-forward + layer norm + residual
5. Building a full Transformer encoder for text classification
6. How this relates to BERT, GPT, and modern LLMs

## 1. The Big Picture

The Transformer paper (Vaswani et al., 2017) replaced RNNs entirely for many tasks.
The key insight: **you don't need recurrence to model sequences** — attention alone is enough.

**Why Transformers beat RNNs:**
- **Parallelizable**: RNNs process step-by-step. Transformers process ALL positions simultaneously.
- **Long-range dependencies**: Attention directly connects any two positions (RNNs lose signal over distance)
- **Better scaling**: More data + more parameters = better results (this is the foundation of GPT-4, etc.)

The Transformer has two variants:
- **Encoder** (BERT): sees the whole sequence, good for classification/understanding
- **Decoder** (GPT): autoregressive, generates one token at a time
- **Encoder-Decoder** (original, T5): for translation/summarization

**We'll build the Encoder.**

## 2. Multi-Head Attention

**Why multiple heads?**

Imagine reading a sentence. At the same time, you're tracking:
- Which noun does "it" refer to? (coreference)
- What is the verb doing? (syntactic role)
- What's the emotional tone? (sentiment)

**Multi-head attention lets the model attend to different aspects simultaneously.**

Each head has its own Q, K, V projection matrices — they learn to ask different questions.
Their outputs are concatenated and projected back to the original dimension.

```python
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) @ W_o
where head_i = Attention(Q @ W_qi, K @ W_ki, V @ W_vi)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.d_k       = embed_dim // num_heads   # dimension per head

        # Single projection matrix for all heads (more efficient than separate)
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):
        B, T, C = x.shape   # batch, seq_len, embed_dim

        # Project to Q, K, V
        Q = self.W_q(x)   # (B, T, C)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape for multi-head: (B, T, C) -> (B, num_heads, T, d_k)
        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention (for all heads in parallel)
        scores   = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights  = self.dropout(F.softmax(scores, dim=-1))
        attended = weights @ V   # (B, num_heads, T, d_k)

        # Concatenate heads: (B, num_heads, T, d_k) -> (B, T, C)
        attended = attended.transpose(1, 2).contiguous().view(B, T, C)

        return self.W_o(attended), weights


# Test
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x   = torch.randn(2, 10, 64)   # (batch=2, seq_len=10, embed=64)
out, attn_w = mha(x)
print(f'Input    : {x.shape}')
print(f'Output   : {out.shape}')
print(f'Attn map : {attn_w.shape}')   # (2, 8, 10, 10) — 8 heads, each is 10x10

## 3. Positional Encoding

Transformers process all positions **in parallel** — they have no inherent sense of order.
Without positional encoding, the model would treat "dog bites man" and "man bites dog" the same.

**Positional encoding** adds a signal to each token embedding that encodes its position.

The original paper uses sine and cosine functions of different frequencies:
```
PE[pos, 2i]   = sin(pos / 10000^(2i/d_model))
PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
```

**Intuition:** Imagine a position clock with many gears running at different speeds.
Each gear (frequency) gives the model different information about where a token is.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, embed_dim)
        pos = torch.arange(max_len).unsqueeze(1).float()   # (max_len, 1)

        # Frequencies for each embedding dimension
        div = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(pos * div)   # even indices
        pe[:, 1::2] = torch.cos(pos * div)   # odd indices

        pe = pe.unsqueeze(0)   # (1, max_len, embed_dim)
        self.register_buffer('pe', pe)   # not a parameter, but saved in state_dict

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


pe = PositionalEncoding(embed_dim=64, max_len=50)

# Visualize the positional encoding patterns
plt.figure(figsize=(14, 5))
pos_enc = pe.pe[0].numpy()   # (50, 64)
plt.imshow(pos_enc, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.xlabel('Embedding Dimension')
plt.ylabel('Position')
plt.title('Positional Encoding — Each row is added to the token embedding at that position')
plt.colorbar()
plt.show()

## 4. The Transformer Encoder Block

Each encoder block contains:
1. **Multi-Head Self-Attention** + **Residual connection** + **Layer Norm**
2. **Feed-Forward Network** (2 linear layers with GELU) + **Residual connection** + **Layer Norm**

**Why residual connections?**
They let gradients flow directly through — this is what allows 12, 24, or even 96 layers to train.

**Why Layer Norm (not Batch Norm)?**
Layer Norm normalizes across the feature dimension (not the batch), so it works with variable-length sequences and batch_size=1.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(embed_dim, num_heads)
        self.ff      = FeedForward(embed_dim, ff_dim, dropout)
        self.norm1   = nn.LayerNorm(embed_dim)
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention with residual connection (pre-norm style)
        attn_out, _ = self.attn(self.norm1(x), mask)
        x = x + self.dropout(attn_out)   # residual

        # Feed-forward with residual connection
        x = x + self.dropout(self.ff(self.norm2(x)))   # residual

        return x


# Test
block = TransformerEncoderBlock(embed_dim=64, num_heads=8, ff_dim=256)
x = torch.randn(4, 20, 64)
out = block(x)
print(f'Block input : {x.shape}')
print(f'Block output: {out.shape}')  # shape preserved

## 5. Full Transformer Text Classifier

Now let's stack everything into a complete model for text classification.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim,
                 num_layers, num_classes, max_len=512, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_enc   = PositionalEncoding(embed_dim, max_len, dropout)

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.norm       = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, mask=None):
        # x: (batch, seq_len) — token indices
        x = self.embedding(x)    # (batch, seq_len, embed_dim)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)

        # Mean pooling over sequence: (batch, seq_len, embed) -> (batch, embed)
        x = x.mean(dim=1)

        return self.classifier(x)


model = TransformerClassifier(
    vocab_size  = 1000,
    embed_dim   = 64,
    num_heads   = 8,
    ff_dim      = 256,
    num_layers  = 3,
    num_classes = 2,
    max_len     = 128,
    dropout     = 0.1,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

x = torch.randint(0, 1000, (16, 50))   # batch=16, seq_len=50
out = model(x)
print(f'Input  : {x.shape}')
print(f'Output : {out.shape}')    # (16, 2)

In [ ]:
# Train on the sentiment dataset from Notebook 10
texts = [
    'i love this movie',      'this film is amazing',    'great acting and story',
    'i really enjoyed it',    'fantastic and wonderful',  'best movie ever seen',
    'absolutely loved it',    'brilliant and inspiring',
    'terrible movie awful',   'waste of time boring',    'hate this film',
    'really bad acting',      'worst movie ever',        'completely disappointed',
    'awful and unwatchable',  'horrible experience',
]
labels = [1]*8 + [0]*8

# Word-level vocabulary
all_words  = sorted(set(' '.join(texts).split()))
word2id    = {w: i+1 for i, w in enumerate(all_words)}
vocab_size = len(word2id) + 1
MAX_LEN    = max(len(t.split()) for t in texts)

def encode(text, word2id, max_len):
    ids = [word2id.get(w, 0) for w in text.split()]
    ids += [0] * (max_len - len(ids))
    return ids[:max_len]

X = torch.tensor([encode(t, word2id, MAX_LEN) for t in texts], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

model     = TransformerClassifier(vocab_size, 32, 4, 128, 2, 2, MAX_LEN, dropout=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    logits = model(X)
    loss   = criterion(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        acc = (logits.argmax(1) == y).float().mean()
        print(f'Epoch {epoch+1:4d}: loss={loss.item():.4f}, acc={acc:.4f}')

plt.plot(losses)
plt.title('Transformer Training Loss')
plt.xlabel('Epoch')
plt.show()

In [ ]:
# Visualize multi-head attention weights
model.eval()

sample_text = 'i love this movie'
x_sample    = torch.tensor([encode(sample_text, word2id, MAX_LEN)], dtype=torch.long)
words_in    = sample_text.split()

# Get attention from first encoder block, first forward pass
embeddings  = model.embedding(x_sample)
embeddings  = model.pos_enc(embeddings)

with torch.no_grad():
    _, attn_weights = model.layers[0].attn(model.layers[0].norm1(embeddings))

attn_w = attn_weights[0].numpy()   # (num_heads, seq_len, seq_len)
num_heads = attn_w.shape[0]

fig, axes = plt.subplots(1, num_heads, figsize=(3 * num_heads, 3))
for i, ax in enumerate(axes):
    T = len(words_in)
    ax.imshow(attn_w[i, :T, :T], cmap='Blues', vmin=0, vmax=attn_w.max())
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels(words_in, rotation=45, fontsize=8)
    ax.set_yticklabels(words_in, fontsize=8)
    ax.set_title(f'Head {i+1}', fontsize=9)
plt.suptitle(f'"{sample_text}" — attention per head', fontsize=11)
plt.tight_layout()
plt.show()

## 6. How This Relates to Modern LLMs

The model we built is essentially a **mini BERT encoder**.

| Model | Architecture | Size | Training |
|-------|-------------|------|---------|
| Our model | Encoder, 2 layers | ~50K params | Tiny sentiment dataset |
| BERT-base | Encoder, 12 layers | 110M params | Masked LM on Wikipedia |
| GPT-4 | Decoder, many layers | ~1.8T params? | Next token prediction on internet |
| Llama 3 70B | Decoder, 80 layers | 70B params | Next token on trillions of tokens |

**The architecture is the same. The scale is different.**

Key differences in modern LLMs:
- **RoPE** or **ALiBi** positional encoding instead of sinusoidal
- **RMSNorm** instead of LayerNorm
- **SwiGLU** instead of GELU in the FFN
- **Grouped-Query Attention** instead of standard multi-head
- **Causal (masked) attention** for autoregressive generation
- Flash Attention for memory-efficient computation

---
## YOUR TURN — Final Capstone Project

**Task:** Build and train a Transformer encoder for multi-class classification.

**Dataset:** Generate 500 synthetic sequences:
- Class 0: sequences starting with high values (> 0)
- Class 1: sequences starting with low values (< 0)
- Class 2: sequences that alternate positive/negative

Use continuous-valued input (regression-style) by treating each timestep as a scalar feature.

**Steps:**
1. Generate 500 sequences of length 20 (3 classes as described)
2. Modify `TransformerClassifier` to accept float inputs (no Embedding layer needed — use `nn.Linear(1, embed_dim)` instead)
3. Add a [CLS] token prepended to each sequence — use the CLS token's output for classification (not mean pooling)
4. Train for 300 epochs with AdamW
5. Achieve > 90% accuracy
6. Visualize the attention maps for one sample from each class — do different heads notice different patterns?

**Why CLS token?** BERT uses a special [CLS] token prepended to every sequence. The CLS token's final representation is used for classification. It "summarizes" the whole sequence through attention.

In [ ]:
# YOUR CODE HERE
import torch

torch.manual_seed(0)

# 1. Generate dataset
def generate_sequences(n, seq_len=20):
    sequences, labels = [], []
    for i in range(n):
        cls = i % 3
        if cls == 0:
            seq = torch.abs(torch.randn(seq_len, 1)) + 0.5    # mostly positive
        elif cls == 1:
            seq = -torch.abs(torch.randn(seq_len, 1)) - 0.5   # mostly negative
        else:
            seq = torch.randn(seq_len, 1)                       # alternating
            seq[::2] = seq[::2].abs()
            seq[1::2] = -seq[1::2].abs()
        sequences.append(seq)
        labels.append(cls)
    return torch.stack(sequences), torch.tensor(labels)

X, y = generate_sequences(500)
print(f'X shape: {X.shape}')   # (500, 20, 1)
print(f'y shape: {y.shape}')   # (500,)

# 2. Transformer with Linear input projection instead of Embedding

# 3. Add CLS token

# 4. Train

# 5. Visualize attention maps